In this tutorial, we are going to evaluate the performance of the naive RAG and the GraphRAG algorithm on a [multi-hop RAG task](https://github.com/yixuantt/MultiHop-RAG).

## Setup
Make sure you install the necessary dependencies by running the following commands:

Import the necessary libraries, and set up your openai api key if needed:

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
#os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"
import json
import sys
sys.path.append("../..")

import nest_asyncio
nest_asyncio.apply()
import logging

logging.basicConfig(level=logging.WARNING)
logging.getLogger("nano-graphrag").setLevel(logging.INFO)
from nano_graphrag import GraphRAG, QueryParam
from datasets import Dataset 
from ragas import evaluate
from ragas.metrics import (
    answer_correctness,
    answer_similarity,
    answer_relevancy
)

c:\Github\nano-graphrag\.conda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Download the dataset from [Github Repo](https://github.com/yixuantt/MultiHop-RAG/tree/main/dataset). 
If should contain two files:
- `MultiHopRAG.json`
- `corpus.json`

After downloading the dataset, replace the below paths to the paths on your machine.

In [3]:

multi_hop_rag_file = "./fixtures/MultiHopRAG.json"
multi_hop_corpus_file = "./fixtures/corpus.json"

## Preprocess

In [4]:

with open(multi_hop_rag_file) as f:
    multi_hop_rag_dataset = json.load(f)
with open(multi_hop_corpus_file) as f:
    multi_hop_corpus = json.load(f)

corups_url_refernces = {}
for cor in multi_hop_corpus:
    corups_url_refernces[cor['url']] = cor

We only use the top-100 queries for evaluation.

In [5]:
multi_hop_rag_dataset = multi_hop_rag_dataset[:100]
print("Queries have types:", set([q['question_type'] for q in multi_hop_rag_dataset]))
total_urls = set()
for q in multi_hop_rag_dataset:
    total_urls.update([up['url'] for up in q['evidence_list']])
corups_url_refernces = {k:v for k, v in corups_url_refernces.items() if k in total_urls}

total_corpus = [f"## {cor['title']}\nAuthor: {cor['author']}, {cor['source']}\nCategory: {cor['category']}\nPublised: {cor['published_at']}\n{cor['body']}" for cor in corups_url_refernces.values()]

print(f"We will need {len(total_corpus)} articles:")
print(total_corpus[0][:200], "...")

Queries have types: {'inference_query', 'temporal_query', 'comparison_query', 'null_query'}
We will need 139 articles:
## ASX set to drop as Wall Street’s September slump deepens
Author: Stan Choe, The Sydney Morning Herald
Category: business
Publised: 2023-09-26T19:11:30+00:00
ETF provider Betashares, which manages $ ...


Add index for the `total_corups` using naive RAG and GraphRAG

In [6]:
import ollama
import numpy as np
from nano_graphrag._utils import compute_args_hash, wrap_embedding_func_with_attrs

# Ollama settings
MODEL = "qwen2.5:14b"  # or any other model you have in Ollama
EMBEDDING_MODEL = "qwen2.5:14b"
EMBEDDING_MODEL_DIM = 5120
EMBEDDING_MODEL_MAX_TOKENS = 8196 #16384

async def ollama_model_if_cache(
    prompt, system_prompt=None, history_messages=[], **kwargs
) -> str:
    kwargs.pop("max_tokens", None)
    kwargs.pop("response_format", None)

    ollama_client = ollama.AsyncClient()
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    hashing_kv = kwargs.pop("hashing_kv", None)
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    
    if hashing_kv is not None:
        args_hash = compute_args_hash(MODEL, messages)
        if_cache_return = await hashing_kv.get_by_id(args_hash)
        if if_cache_return is not None:
            return if_cache_return["return"]
            
    response = await ollama_client.chat(model=MODEL, messages=messages, **kwargs)
    result = response["message"]["content"]
    #result = result.split("</think>")[-1] # 去掉<think>
    if hashing_kv is not None:
        await hashing_kv.upsert({args_hash: {"return": result, "model": MODEL}})
    return result

@wrap_embedding_func_with_attrs(
    embedding_dim=EMBEDDING_MODEL_DIM,
    max_token_size=EMBEDDING_MODEL_MAX_TOKENS,
)
async def ollama_embedding(texts: list[str]) -> np.ndarray:
    embed_text = []
    for text in texts:
        data = ollama.embeddings(model=EMBEDDING_MODEL, prompt=text)
        embedding = np.array(data["embedding"])
        # Ensure the embedding dimension matches
        assert embedding.shape[0] == EMBEDDING_MODEL_DIM, f"Expected dimension {EMBEDDING_MODEL_DIM}, got {embedding.shape[0]}"
        embed_text.append(embedding)
    return np.vstack(embed_text)



In [7]:
# First time indexing will cost many time, roughly 15~20 minutes
from nano_graphrag._llm import openai_complete_if_cache  # 添加这行
from typing import Optional, List
from nano_graphrag._utils import compute_args_hash

graphrag_func = GraphRAG(
    working_dir="nano_graphrag_cache_multihop_rag_test_qwen25_14b",
    enable_naive_rag=True,
    embedding_func_max_async=4,
    embedding_batch_num=64,
    best_model_func=ollama_model_if_cache,
    cheap_model_func=ollama_model_if_cache,
    embedding_func=ollama_embedding
)

graphrag_func.insert(total_corpus)

INFO:nano-graphrag:Creating working directory nano_graphrag_cache_multihop_rag_test_qwen25_14b
INFO:nano-graphrag:Load KV full_docs with 0 data
INFO:nano-graphrag:Load KV text_chunks with 0 data
INFO:nano-graphrag:Load KV llm_response_cache with 0 data
INFO:nano-graphrag:Load KV community_reports with 0 data
INFO:nano-graphrag:[New Docs] inserting 139 docs
INFO:nano-graphrag:[New Chunks] inserting 408 chunks
INFO:nano-graphrag:Insert chunks for naive RAG
INFO:nano-graphrag:Inserting 408 vectors to chunks
INFO:nano-graphrag:[Entity Extraction]...


INFO:nano-graphrag:Inserting 422 vectors to entities


INFO:nano-graphrag:[Community Report]...
INFO:nano-graphrag:Each level has communities: {0: 3}
INFO:nano-graphrag:Generating by levels: [0]
INFO:nano-graphrag:JSON data successfully extracted.


INFO:nano-graphrag:JSON data successfully extracted.


INFO:nano-graphrag:JSON data successfully extracted.


INFO:nano-graphrag:Writing graph with 447 nodes, 208 edges


Look at the response of different RAG methods on the first query:

In [8]:
response_formate = "Single phrase or sentence, concise and no redundant explanation needed. If you don't have the answer in context, Just response 'Insufficient information'"
naive_rag_query_param = QueryParam(mode='naive', response_type=response_formate)
naive_rag_query_only_context_param = QueryParam(mode='naive', only_need_context=True)
local_graphrag_query_param = QueryParam(mode='local', response_type=response_formate)
local_graphrag_only_context__param = QueryParam(mode='local', only_need_context=True)
global_graphrag_query_param = QueryParam(mode='global', response_type=response_formate)
global_graphrag_only_context__param = QueryParam(mode='global', only_need_context=True)

In [9]:
query = multi_hop_rag_dataset[0]
print("Question:", query['query'])
print("GroundTruth Answer:", query['answer'])

Question: Who is the individual associated with the cryptocurrency industry facing a criminal trial on fraud and conspiracy charges, as reported by both The Verge and TechCrunch, and is accused by prosecutors of committing fraud for personal gain?
GroundTruth Answer: Sam Bankman-Fried


In [10]:
print("NaiveRAG Answer:", graphrag_func.query(query['query'], param=naive_rag_query_param))

INFO:nano-graphrag:Truncate 20 to 19 chunks


NaiveRAG Answer: The individual you are referring to is likely Evro Peizer, but it seems there might be some confusion in the name. If we go by recent high-profile cases in the cryptocurrency industry, one prominent figure who fits this description is Samuel Epstein, also known as "Samuel Zeller." However, the most well-known case fitting your description involves Eric Toussaint and others associated with BitConnect but the primary individual often highlighted in major media outlets for a similar nature of charges is generally someone like Faiyaz Jaleel or even related to Bitconnect's operator but the exact named person being currently on trial as per your query might be Samuel Epstein though he isn't widely publicized under his real name.

If you are referring to a more recent and highly publicized case, it may be important to clarify since multiple figures in cryptocurrency history have faced legal troubles. One of the most recently high-profile cases involving fraud charges is that 

In [11]:
print("Local GraphRAG Answer:", graphrag_func.query(query['query'], param=local_graphrag_query_param))

INFO:nano-graphrag:Using 20 entites, 0 communities, 18 relations, 14 text units


Local GraphRAG Answer: The person you are referring to is likely Ethan Linderman. However, one of the most high-profile cases in recent years involves Ievgeniy Viktorovych Bogachev, better known as "Lucky1234," or more famously others like Roger Ver and Jed McCoppens but the most talked about recently is Eric Trump's chief financial officer, Alan Brian Friedberg but none of them match exactly with the context. The case you are probably referring to in recent reporting by sources such as The Verge and TechCrunch is that of Samuel Franklin "Sam" Bankman-Fried (SBF), founder of FTX cryptocurrency exchange. However, your detailed description points towards a lesser spotlight figure named Terraform Labs' CEO Do Kwon or potentially others involved in the FTX scandal but most exactly matches Sam Bankman-Fried's case as he was charged with conspiracy and fraud by U.S prosecutors for personal gain at the expense of investors and users.

Sam Bankman-Fried founded Alameda Research in 2017 and the

In [12]:
print("Global GraphRAG Answer:", graphrag_func.query(query['query'], param=global_graphrag_query_param))

INFO:nano-graphrag:Revtrieved 3 communities
INFO:nano-graphrag:Grouping to 1 groups for global search
INFO:nano-graphrag:JSON data successfully extracted.


Global GraphRAG Answer: Sorry, I'm not able to provide an answer to that question.


Great! Now we're ready to evaluate more detailed metrics. We will use [ragas](https://docs.ragas.io/en/stable/) to evalue the answers' quality.

In [13]:
questions = [q['query'] for q in multi_hop_rag_dataset]
labels = [q['answer'] for q in multi_hop_rag_dataset]

In [14]:
from tqdm import tqdm
logging.getLogger("nano-graphrag").setLevel(logging.WARNING)

naive_rag_answers = [
    graphrag_func.query(q, param=naive_rag_query_param) for q in tqdm(questions)
]

100%|██████████| 100/100 [09:35<00:00,  5.75s/it]


In [15]:
local_graphrag_answers = [
    graphrag_func.query(q, param=local_graphrag_query_param) for q in tqdm(questions)
]

100%|██████████| 100/100 [09:47<00:00,  5.87s/it]


In [16]:
global_graphrag_answers = [
    graphrag_func.query(q, param=global_graphrag_query_param) for q in tqdm(questions)
]

100%|██████████| 100/100 [02:58<00:00,  1.78s/it]


In [31]:
from ragas.llms import LangchainLLMWrapper
from langchain.llms import Ollama
from langchain.chat_models import ChatOpenAI
from langchain.callbacks.manager import CallbackManager
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from ragas.run_config import RunConfig
# Configure OpenRouter with DeepSeek-v3
deepseek_llm = ChatOpenAI(
    model="deepseek/deepseek-chat:free", 
    temperature=0,  # Important for consistent evaluation
    max_tokens=2048,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)
run_config = RunConfig(
    max_retries=10,
    max_wait=60,
    log_tenacity=True,
    max_workers = 4,
)
# Configure the LLM wrapper with specific parameters
llm = LangchainLLMWrapper(
    deepseek_llm,
)

# Evaluate using DeepSeek-v3 with executor config
naive_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": naive_rag_answers,
    }),
    metrics=[
        answer_correctness,
        answer_similarity,
    ],
    llm=llm,
    run_config=run_config,
)

Evaluating:   2%|▏         | 3/200 [00:01<02:23,  1.37it/s]ERROR:ragas.executor:Exception raised in Job[4]: OutputParserException(Failed to parse StringIO from completion {"statements": ["The individual referred to is likely Sam Altman.", "Sam Altman served as the CEO of OpenAI.", "Sam Altman's departure from OpenAI was considered shocking by many in the tech community.", "Outlets like Fortune and TechCrunch considered Sam Altman's departure shocking.", "Speculations and theories existed regarding Sam Altman's relationship with the board.", "Speculations and theories existed regarding the transparency of Sam Altman's actions while leading OpenAI.", "Speculations and theories particularly concerned the direction and oversight of generative AI technologies like ChatGPT.", "Any specific claims about a lack of truthfulness would be speculative without concrete evidence or official statements from OpenAI\u2019s leadership.", "The tech industry often has discussions and speculations around m

In [32]:
local_graphrag_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": local_graphrag_answers,
    }),
    metrics=[
        #answer_relevancy,
        answer_correctness,
        answer_similarity,
    ],
    llm=llm,
    run_config=run_config,
)

Evaluating:   0%|          | 1/200 [00:00<01:04,  3.09it/s]ERROR:ragas.executor:Exception raised in Job[0]: TypeError('NoneType' object is not iterable)
ERROR:ragas.executor:Exception raised in Job[4]: TypeError('NoneType' object is not iterable)
Evaluating:  20%|██        | 40/200 [00:06<00:27,  5.83it/s]ERROR:ragas.executor:Exception raised in Job[42]: TypeError('NoneType' object is not iterable)
ERROR:ragas.executor:Exception raised in Job[44]: TypeError('NoneType' object is not iterable)
Evaluating:  26%|██▌       | 52/200 [00:08<00:22,  6.64it/s]ERROR:ragas.executor:Exception raised in Job[54]: TypeError('NoneType' object is not iterable)
ERROR:ragas.executor:Exception raised in Job[56]: TypeError('NoneType' object is not iterable)
Evaluating:  36%|███▌      | 71/200 [00:10<00:13,  9.68it/s]ERROR:ragas.executor:Exception raised in Job[72]: TypeError('NoneType' object is not iterable)
ERROR:ragas.executor:Exception raised in Job[74]: TypeError('NoneType' object is not iterable)
Eva

In [33]:
global_graphrag_results = evaluate(
    Dataset.from_dict({
        "question": questions,
        "ground_truth": labels,
        "answer": global_graphrag_answers,
    }),
    metrics=[
        #answer_relevancy,
        answer_correctness,
        answer_similarity,
    ],
    llm=llm,
    run_config=run_config,
)

Evaluating:   0%|          | 1/200 [00:00<01:27,  2.26it/s]ERROR:ragas.executor:Exception raised in Job[0]: TypeError('NoneType' object is not iterable)
ERROR:ragas.executor:Exception raised in Job[4]: TypeError('NoneType' object is not iterable)
Evaluating:   8%|▊         | 17/200 [00:02<00:23,  7.93it/s]ERROR:ragas.executor:Exception raised in Job[18]: TypeError('NoneType' object is not iterable)
ERROR:ragas.executor:Exception raised in Job[20]: TypeError('NoneType' object is not iterable)
Evaluating:  12%|█▏        | 23/200 [00:03<00:21,  8.28it/s]ERROR:ragas.executor:Exception raised in Job[24]: TypeError('NoneType' object is not iterable)
ERROR:ragas.executor:Exception raised in Job[26]: TypeError('NoneType' object is not iterable)
Evaluating:  14%|█▍        | 29/200 [00:03<00:19,  8.87it/s]ERROR:ragas.executor:Exception raised in Job[30]: TypeError('NoneType' object is not iterable)
ERROR:ragas.executor:Exception raised in Job[32]: TypeError('NoneType' object is not iterable)
Eva

In [34]:
print("Naive RAG results", naive_results)
print("Local GraphRAG results", local_graphrag_results)
print("global GraphRAG results", global_graphrag_results)

Naive RAG results {'answer_correctness': nan, 'semantic_similarity': 0.7573}
Local GraphRAG results {'answer_correctness': nan, 'semantic_similarity': 0.7544}
global GraphRAG results {'answer_correctness': nan, 'semantic_similarity': 0.7756}
